In [ ]:
# Ensure dependencies are synced
!uv sync

# Primary Topic Classification

This notebook builds a classifier to predict the primary topic of tweets using the tweets_v1.csv dataset.

## Approach
- Load and explore the dataset
- Analyze topic distribution
- Train a TF-IDF + Logistic Regression classifier
- Train a transformer-based classifier (DistilBERT)
- Compare model performance
- Create prediction functions for new tweets

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load and Explore Data

In [ ]:
# Load the tweets dataset
df = pd.read_csv('../../sentiment_classification/data/tweets_v1.csv')

# Display basic information
print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nFirst 5 rows:")
df.head()

In [ ]:
# Explore primary topic distribution
print("Primary Topic Distribution:")
topic_counts = df['primary_topic'].value_counts()
print(topic_counts)
print(f"\nNumber of unique topics: {df['primary_topic'].nunique()}")
print(f"\nPercentage distribution:")
print((topic_counts / len(df) * 100).round(2))

In [ ]:
# Visualize topic distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart
topic_counts.plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Primary Topic Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Topic', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].tick_params(axis='x', rotation=45)

# Pie chart (top 10 topics)
top_10_topics = topic_counts.head(10)
axes[1].pie(top_10_topics.values, labels=top_10_topics.index, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Top 10 Topics Distribution', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Sample tweets from different topics
print("Sample tweets by topic (top 5 topics):\n")
for topic in topic_counts.head(5).index:
    print(f"\n{'='*80}")
    print(f"TOPIC: {topic.upper()}")
    print(f"{'='*80}")
    samples = df[df['primary_topic'] == topic]['tweet'].head(3)
    for i, text in enumerate(samples, 1):
        print(f"{i}. {text}")

## 2. Traditional ML Approach: TF-IDF + Logistic Regression

In [ ]:
# Prepare data for training
X = df['tweet']
y = df['primary_topic']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")
print(f"\nTopic distribution in training set:")
print(y_train.value_counts())

In [ ]:
# Create TF-IDF vectorizer
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.9
)

# Transform text to TF-IDF features
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f"TF-IDF feature matrix shape: {X_train_tfidf.shape}")
print(f"Number of features: {len(vectorizer.get_feature_names_out())}")

In [ ]:
# Train Logistic Regression classifier
print("Training Logistic Regression model...")
clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train_tfidf, y_train)
print("Training complete!")

# Make predictions
y_pred = clf.predict(X_test_tfidf)

# Evaluate
accuracy = accuracy_score(y_test, y_pred)
print(f"\nTest Accuracy: {accuracy:.4f}")

In [ ]:
# Detailed classification report
print("Classification Report:\n")
print(classification_report(y_test, y_pred))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=clf.classes_, yticklabels=clf.classes_)
plt.title('Confusion Matrix - Topic Classification', fontsize=14, fontweight='bold')
plt.ylabel('True Topic', fontsize=12)
plt.xlabel('Predicted Topic', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 3. Feature Importance Analysis

In [ ]:
# Get top features for each topic
feature_names = vectorizer.get_feature_names_out()
n_top_features = 10

print("Top features (words/bigrams) for each topic:\n")
for i, topic in enumerate(clf.classes_):
    top_indices = clf.coef_[i].argsort()[-n_top_features:][::-1]
    top_features = [feature_names[idx] for idx in top_indices]
    print(f"\n{topic.upper()}:")
    print(", ".join(top_features))

## 4. Prediction Function for New Tweets

In [ ]:
def classify_topic(tweet_text, return_probabilities=False):
    """
    Classify the primary topic of a tweet.
    
    Args:
        tweet_text (str): The tweet text to classify
        return_probabilities (bool): If True, return probabilities for all topics
    
    Returns:
        str or dict: Predicted topic or dict with topic and probabilities
    """
    # Transform the text
    tweet_tfidf = vectorizer.transform([tweet_text])
    
    # Predict
    predicted_topic = clf.predict(tweet_tfidf)[0]
    
    if return_probabilities:
        # Get probabilities for all topics
        probabilities = clf.predict_proba(tweet_tfidf)[0]
        topic_probs = dict(zip(clf.classes_, probabilities))
        # Sort by probability
        topic_probs = dict(sorted(topic_probs.items(), key=lambda x: x[1], reverse=True))
        return {
            'predicted_topic': predicted_topic,
            'probabilities': topic_probs
        }
    
    return predicted_topic

def classify_topics_batch(tweet_texts):
    """
    Classify topics for multiple tweets.
    
    Args:
        tweet_texts (list): List of tweet texts
    
    Returns:
        list: List of predicted topics
    """
    tweets_tfidf = vectorizer.transform(tweet_texts)
    predictions = clf.predict(tweets_tfidf)
    return predictions.tolist()

print("Prediction functions created successfully!")

In [ ]:
# Test the prediction function with new examples
test_tweets = [
    "Just had an amazing burger at the new restaurant downtown!",
    "The traffic on the highway is absolutely terrible today",
    "My cat just did the cutest thing ever!",
    "This new movie is absolutely fantastic, best I've seen all year",
    "Climate change is becoming more urgent every day"
]

print("Test Predictions:\n")
for tweet in test_tweets:
    result = classify_topic(tweet, return_probabilities=True)
    print(f"\nTweet: {tweet}")
    print(f"Predicted Topic: {result['predicted_topic']}")
    print("Top 3 Probabilities:")
    for topic, prob in list(result['probabilities'].items())[:3]:
        print(f"  {topic}: {prob:.4f}")

## 5. Transformer-Based Approach (Optional)

Using DistilBERT for more advanced topic classification

In [ ]:
# Transformer-based approach
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
# Prepare data for transformer
# Create label mapping
unique_topics = sorted(df['primary_topic'].unique())
topic2id = {topic: idx for idx, topic in enumerate(unique_topics)}
id2topic = {idx: topic for topic, idx in topic2id.items()}

print(f"Number of topics: {len(unique_topics)}")
print(f"\nTopic to ID mapping (first 5):")
for i, (topic, idx) in enumerate(list(topic2id.items())[:5]):
    print(f"  {topic}: {idx}")

In [ ]:
# Convert to datasets
train_data = pd.DataFrame({
    'text': X_train.values,
    'label': y_train.map(topic2id).values
})

test_data = pd.DataFrame({
    'text': X_test.values,
    'label': y_test.map(topic2id).values
})

train_dataset = Dataset.from_pandas(train_data)
test_dataset = Dataset.from_pandas(test_data)

print(f"Train dataset: {len(train_dataset)} samples")
print(f"Test dataset: {len(test_dataset)} samples")

In [ ]:
# Initialize tokenizer
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=128)

# Tokenize datasets
train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

print("Tokenization complete!")

In [ ]:
# Initialize model
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=len(unique_topics)
)

# Training arguments
training_args = TrainingArguments(
    output_dir='../models/distilbert_topic_classifier',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=50,
    weight_decay=0.01,
    logging_dir='../models/logs',
    logging_steps=10,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
)

# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

print("Model initialized. Ready for training!")
print("\nNote: Training can take several minutes. Uncomment the next cell to train.")

In [ ]:
# Uncomment to train the transformer model
# print("Starting training...")
# trainer.train()
# print("\nTraining complete!")

# # Evaluate
# eval_results = trainer.evaluate()
# print(f"\nEvaluation results:")
# for key, value in eval_results.items():
#     print(f"  {key}: {value:.4f}")

## 6. Save Models

In [ ]:
# Save the TF-IDF model and vectorizer
import pickle

# Save vectorizer
with open('../models/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

# Save classifier
with open('../models/topic_classifier.pkl', 'wb') as f:
    pickle.dump(clf, f)

# Save topic mappings
with open('../models/topic_mappings.pkl', 'wb') as f:
    pickle.dump({'topic2id': topic2id, 'id2topic': id2topic}, f)

print("Models saved successfully!")
print("\nSaved files:")
print("  - ../models/tfidf_vectorizer.pkl")
print("  - ../models/topic_classifier.pkl")
print("  - ../models/topic_mappings.pkl")

## Summary

### Model Performance
- **TF-IDF + Logistic Regression**: Fast, interpretable, good baseline performance
- **DistilBERT** (optional): More accurate but requires more computational resources

### Key Features
- Multi-class classification for primary topics
- Probability distributions for predictions
- Batch prediction support
- Saved models for production use

### Next Steps
1. Use the saved models in `main.py` for production predictions
2. Experiment with different feature engineering approaches
3. Try ensemble methods combining multiple models
4. Add secondary topic prediction capability